# EasyRAG Eval-Driven Debugging

## Chapter position

This chapter closes the loop. It separates retrieval quality from answer quality and shows how to turn runtime behavior into measurements you can actually debug.

## Learning question

How does evaluation point you to the failing stage instead of only telling you that a score is low?

## Success criteria

- start from a weak evaluation result
- inspect the failing case and its retrieval metadata
- change parameters and rerun the report to confirm the fix direction

## Source code anchors

- `easyrag.rag.evaluation.evaluate_retrieval`
- `easyrag.rag.types.QueryParam`
- `easyrag.rag.types.EvalCase`
- `easyrag.rag.retrieval.pipeline.execute_query`


## Method principles

- `easyrag.rag.evaluation.evaluate_retrieval`: This package-level entry point runs the retrieval evaluator over a case set and returns both case-level reports and aggregate metrics.
- `easyrag.rag.types.QueryParam`: This dataclass is the retrieval control surface. It does not run retrieval itself; it carries mode, top-k, rewrite, MQE, rerank, and filter policy into the pipeline.
- `easyrag.rag.types.EvalCase`: This dataclass defines one evaluation expectation. It ties a question to expected documents, snippets, abstention behavior, and optional reference text so evaluation stays deterministic.
- `easyrag.rag.retrieval.pipeline.execute_query`: This is the end-to-end retrieval pipeline. It runs preparation, candidate generation, filtering, hydration, optional rerank, and result assembly into one `QueryResult`.

Design reason: these anchors keep runtime behavior and scoring logic close together. The notebook uses them so that a metric report still points back to concrete cases, retrieved evidence, and explicit expectations instead of becoming a detached benchmark number.


## How the code fits together

The flow in this notebook is bad metric -> failing query metadata -> recovered configuration. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

Design reason: the notebook keeps case construction, per-case evidence, and aggregate metrics in one visible chain. That pacing prevents evaluation from becoming a black-box score generator and makes regression reports easier to audit.


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.evaluation import evaluate_retrieval

## What this cell is proving

We deliberately start with a bad parameter choice: a score threshold that filters out the very evidence we care about.

Design reason: the notebook turns evaluation into explicit cases, flags, and reports here so every score remains tied to visible evidence. That is why the later metrics can be challenged, recomputed, or debugged case by case.


In [ ]:
debug_tmp = tempfile.TemporaryDirectory()
debug_rag = EasyRAG(
    working_dir=debug_tmp.name,
    workspace="eval-debug-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(debug_rag.initialize_storages())
run_sync(
    debug_rag.ainsert(
        [
            "# Architecture\nretrieval retrieval retrieval keeps architecture grounded.\n",
            "# Guide\nretrieval keeps guides readable.\n",
        ],
        ids=["doc::architecture", "doc::guide"],
        file_paths=["docs/architecture.md", "docs/guide.md"],
    )
)
case = EvalCase(
    question="Which document keeps architecture grounded?",
    expected_document_ids=("doc::architecture",),
)
bad_param = QueryParam(
    mode="naive",
    rewrite_enabled=False,
    mqe_enabled=False,
    min_score=130.0,
    chunk_top_k=3,
)
good_param = QueryParam(
    mode="naive",
    rewrite_enabled=False,
    mqe_enabled=False,
    min_score=None,
    chunk_top_k=3,
)
bad_report = run_sync(evaluate_retrieval(debug_rag, [case], bad_param))
good_report = run_sync(evaluate_retrieval(debug_rag, [case], good_param))
_print_json(
    {"bad_metrics": bad_report["metrics"], "good_metrics": good_report["metrics"]}
)

## Why this output looks like this

The next cell pulls the failing query directly so you can inspect the retrieval metadata that explains the weak score.

Design reason: the output keeps evidence and metadata together because retrieval is not only about the top text string. The notebook preserves ranking context, mode choice, and query-preparation fields so the next step can explain why these citations appeared and why others did not.


In [ ]:
failing_result = run_sync(debug_rag.aquery(case.question, bad_param))
recovered_result = run_sync(debug_rag.aquery(case.question, good_param))
_print_json(
    {
        "failing_metadata": failing_result.metadata,
        "failing_citations": failing_result.citations,
        "recovered_metadata": recovered_result.metadata,
        "recovered_citations": recovered_result.citations,
    }
)

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

The optional path repeats the same pattern with provider-backed retrieval when available: evaluate first, inspect the failing case second.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(
        working_dir=provider_tmp.name, workspace="provider-eval-debug-demo"
    )
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                "# Retrieval\nGrounded retrieval keeps answers inspectable.\n",
                ids=["doc::retrieval"],
                file_paths=["docs/retrieval.md"],
            )
        )
        provider_report = run_sync(
            evaluate_retrieval(
                provider_rag,
                [
                    EvalCase(
                        question="What keeps answers inspectable?",
                        expected_document_ids=("doc::retrieval",),
                    )
                ],
                QueryParam(mode="hybrid", chunk_top_k=2),
            )
        )
        _print_json(provider_report)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()

## Common mistakes / debugging cues

- A single metric rarely tells you which stage broke. Keep retrieval and answer quality separate.
- Eval cases should be small enough to audit by hand. If the expectation is vague, the metric will be vague too.
- Use metrics to narrow the search space, then inspect the case-level reports and runtime metadata.

## Takeaway

The metric alone only told us that retrieval was weak. The metadata told us why: an overly strict score threshold filtered away relevant evidence. Eval-driven debugging works when you move from the bad score to the specific stage signal that explains it.

In [ ]:
run_sync(debug_rag.finalize_storages())
debug_tmp.cleanup()
print("Cleaned up the eval-driven-debugging workspace.")

## Where to go next

- Continue with [06_06_latency_cost_and_regression_checks.ipynb](06_06_latency_cost_and_regression_checks.ipynb) to extend evaluation beyond correctness into operational checks.
- Read [07-optimization-overview.md](../../docs/07-optimization-overview.md) to see how these evaluation findings feed optimization.
- Revisit [04_07_retrieval_failure_cases_and_debugging.ipynb](../04_retrieval/04_07_retrieval_failure_cases_and_debugging.ipynb) if you want the retrieval-only debugging view.